# Підключення бібліотек

In [25]:
import pandas as pd
import numpy as np
import pyarrow as pa

# Зчитування даних

In [42]:
anime = pd.read_csv("anime_with_synopsis.csv")
ratings = pd.read_csv("rating_complete.csv")

print("anime:", anime.shape)
print("ratings:", ratings.shape)

display(anime.head())
display(ratings.head())

anime: (16214, 5)
ratings: (57633278, 3)


,MAL_ID,Name,Score,Genres,sypnopsis
0,1,Cowboy Bebop,8.78,"Action, Adventure, Comedy, Drama, Sci-Fi, Space","In the year 2071, humanity has colonized sever..."
1,5,Cowboy Bebop: Tengoku no Tobira,8.39,"Action, Drama, Mystery, Sci-Fi, Space","other day, another bounty—such is the life of ..."
2,6,Trigun,8.24,"Action, Sci-Fi, Adventure, Comedy, Drama, Shounen","Vash the Stampede is the man with a $$60,000,0..."
3,7,Witch Hunter Robin,7.27,"Action, Mystery, Police, Supernatural, Drama, ...",ches are individuals with special powers like ...
4,8,Bouken Ou Beet,6.98,"Adventure, Fantasy, Shounen, Supernatural",It is the dark century and the people are suff...


,user_id,anime_id,rating
0,0,430,9
1,0,1004,5
2,0,3010,7
3,0,570,7
4,0,2762,9


In [3]:
print(anime.dtypes)
print(ratings.dtypes)

MAL_ID       int64
Name           str
Score          str
Genres         str
sypnopsis      str
dtype: object
user_id     int64
anime_id    int64
rating      int64
dtype: object


In [4]:
print(anime.dtypes)
print(ratings.dtypes)

MAL_ID       int64
Name           str
Score          str
Genres         str
sypnopsis      str
dtype: object
user_id     int64
anime_id    int64
rating      int64
dtype: object


In [43]:
print("Unique anime IDs:", anime["MAL_ID"].nunique())
print("Unique rated anime IDs:", ratings["anime_id"].nunique())

print(
    "Ratings without matching anime:",
    (~ratings["anime_id"].isin(anime["MAL_ID"])).sum()
)

Unique anime IDs: 16214
Unique rated anime IDs: 16872
Ratings without matching anime: 906417


In [44]:
# ID аніме, які присутні в ratings
rated_anime_ids = ratings['anime_id'].unique()

# Залишаємо в anime тільки аніме, які мають оцінки
anime_filtered = anime[anime['MAL_ID'].isin(rated_anime_ids)].copy()

# Залишаємо в ratings тільки оцінки аніме, які є в anime
ratings_filtered = ratings[ratings['anime_id'].isin(anime_filtered['MAL_ID'])].copy()


In [45]:
print(anime_filtered.shape)
print(ratings_filtered.shape)

(15550, 5)
(56726861, 3)


In [46]:
user_counts = ratings_filtered["user_id"].value_counts()
anime_counts = ratings_filtered["anime_id"].value_counts()

In [47]:
valid_users = user_counts[user_counts >= 20].index
valid_anime = anime_counts[anime_counts >= 50].index

In [48]:
ratings_filtered = ratings_filtered[
    ratings_filtered["user_id"].isin(valid_users) &
    ratings_filtered["anime_id"].isin(valid_anime)
].copy()

In [49]:
print("Original:", ratings.shape)
print("Filtered:", ratings_filtered.shape)

print("Users:", ratings_filtered["user_id"].nunique())
print("Anime:", ratings_filtered["anime_id"].nunique())

Original: (57633278, 3)
Filtered: (56273090, 3)
Users: 264537
Anime: 10861


# Обєднання датасетів

In [50]:
# Interactions
interactions = ratings_filtered[
    ["user_id", "anime_id", "rating"]
].copy()

# Item metadata
items = anime[
    [
        "MAL_ID",
        "Name",
        "Score",
        "Genres",
        "sypnopsis"
    ]
].copy()

items = items.rename(columns={
    "MAL_ID": "anime_id"
})

# Залишаємо тільки потрібні тайтли
items = items[
    items["anime_id"].isin(
        interactions["anime_id"].unique()
    )
].copy()

print("Interactions:", interactions.shape)
print("Items:", items.shape)

Interactions: (56273090, 3)
Items: (10861, 5)


In [51]:
print(interactions.memory_usage(deep=True).sum() / 1024**3, "GB")
print(items.memory_usage(deep=True).sum() / 1024**3, "GB")

1.6770687699317932 GB
0.005589696578681469 GB


In [52]:
interactions["user_id"] = interactions["user_id"].astype("int32")
interactions["anime_id"] = interactions["anime_id"].astype("int16")
interactions["rating"] = interactions["rating"].astype("int8")

In [53]:
print(
    interactions.memory_usage(deep=True).sum() / 1024**2,
    "MB"
)

804.9930095672607 MB


In [54]:
print("pandas:", pd.__version__)
print("pyarrow:", pa.__version__)

pandas: 3.0.5
pyarrow: 25.0.1


In [55]:
interactions.to_parquet(
    "interactions.parquet",
    engine="pyarrow",
    compression="snappy",
    index=False
)

In [56]:
items.to_parquet(
    "items.parquet",
    engine="pyarrow",
    compression="snappy",
    index=False
)

In [57]:
test_interactions = pd.read_parquet("interactions.parquet")
test_items = pd.read_parquet("items.parquet")

print(test_interactions.shape)
print(test_items.shape)

(56273090, 3)
(10861, 5)


In [58]:
user_counts = interactions["user_id"].value_counts()
anime_counts = interactions["anime_id"].value_counts()

print("=== USERS ===")
print(user_counts.describe())

print("\n=== ANIME ===")
print(anime_counts.describe())

=== USERS ===
count    264537.000000
mean        212.722946
std         249.957025
min          18.000000
25%          69.000000
50%         139.000000
75%         265.000000
max       10362.000000
Name: count, dtype: float64

=== ANIME ===
count     10861.000000
mean       5181.207071
std       12336.022806
min          49.000000
25%         189.000000
50%         804.000000
75%        4193.000000
max      174247.000000
Name: count, dtype: float64


In [59]:
for threshold in [10, 20, 50, 100]:
    print(
        f"Users with >= {threshold} ratings:",
        (user_counts >= threshold).sum()
    )

for threshold in [20, 50, 100, 500]:
    print(
        f"Anime with >= {threshold} ratings:",
        (anime_counts >= threshold).sum()
    )

Users with >= 10 ratings: 264537
Users with >= 20 ratings: 264532
Users with >= 50 ratings: 222751
Users with >= 100 ratings: 166443
Anime with >= 20 ratings: 10861
Anime with >= 50 ratings: 10859
Anime with >= 100 ratings: 9321
Anime with >= 500 ratings: 6333


In [60]:
MIN_USER_RATINGS = 20
MIN_ANIME_RATINGS = 100

filtered_interactions = interactions.copy()

previous_shape = None

for iteration in range(10):
    user_counts = filtered_interactions["user_id"].value_counts()
    valid_users = user_counts[
        user_counts >= MIN_USER_RATINGS
    ].index

    filtered_interactions = filtered_interactions[
        filtered_interactions["user_id"].isin(valid_users)
    ]

    anime_counts = filtered_interactions["anime_id"].value_counts()
    valid_anime = anime_counts[
        anime_counts >= MIN_ANIME_RATINGS
    ].index

    filtered_interactions = filtered_interactions[
        filtered_interactions["anime_id"].isin(valid_anime)
    ]

    current_shape = filtered_interactions.shape

    print(
        f"Iteration {iteration + 1}: {current_shape}"
    )

    if current_shape == previous_shape:
        break

    previous_shape = current_shape

Iteration 1: (56163513, 3)
Iteration 2: (56163348, 3)
Iteration 3: (56163348, 3)


In [61]:
print("Final shape:", filtered_interactions.shape)

print(
    "Minimum user ratings:",
    filtered_interactions.groupby("user_id").size().min()
)

print(
    "Minimum anime ratings:",
    filtered_interactions.groupby("anime_id").size().min()
)

Final shape: (56163348, 3)
Minimum user ratings: 20
Minimum anime ratings: 100


In [62]:
valid_anime_ids = filtered_interactions["anime_id"].unique()

filtered_items = items[
    items["anime_id"].isin(valid_anime_ids)
].copy()

print("Interactions:", filtered_interactions.shape)
print("Items:", filtered_items.shape)

Interactions: (56163348, 3)
Items: (7189, 5)


In [63]:
print(
    "Min user ratings:",
    filtered_interactions.groupby("user_id").size().min()
)

print(
    "Min anime ratings:",
    filtered_interactions.groupby("anime_id").size().min()
)

Min user ratings: 20
Min anime ratings: 100


In [64]:
filtered_interactions.to_parquet(
    "filtered_interactions.parquet",
    engine="pyarrow",
    compression="snappy",
    index=False
)

filtered_items.to_parquet(
    "filtered_items.parquet",
    engine="pyarrow",
    compression="snappy",
    index=False
)